# LaTeX-to-Abstract Generation Challenge, by Tim Tarver

# Load Dataset

In [ ]:
import pandas as pd
import warnings
!python -Xfrozen_modules=off -m jupyter notebook
warnings.filterwarnings("ignore", category=SyntaxWarning)

train = pd.read_csv("dataset/public/train.csv")
test = pd.read_csv("dataset/public/test.csv")

print(len(train), len(test))

# LaTeX Cleaning

In [ ]:
import re

def clean_latex(text):

    text = re.sub(r'\\cite\{.*?\}', '', text)
    text = re.sub(r'\\ref\{.*?\}', '', text)
    text = re.sub(r'\\label\{.*?\}', '', text)

    text = re.sub(r'\\begin\{.*?\}', '', text)
    text = re.sub(r'\\end\{.*?\}', '', text)

    text = re.sub(r'\\[a-zA-Z]+', '', text)

    text = re.sub(r'\$.*?\$', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

train["latex"] = train["latex"].apply(clean_latex)
test["latex"] = test["latex"].apply(clean_latex)

# Section Extraction 
# Most abstracts summarize:

# * problem

# * method

# * results

# * We extract early sections.

In [ ]:
def extract_key_content(text, max_words=1200):

    words = text.split()

    if len(words) <= max_words:
        return text

    return " ".join(words[:max_words])

# Prompt Engineering 

In [ ]:
def build_prompt(latex):

    content = extract_key_content(latex)

    prompt = f"""
Write a scientific paper abstract based on the following manuscript content.

The abstract must:
- summarize the research problem
- describe the proposed method
- mention key results or contributions
- be concise (120–180 words)

Manuscript:
{content}

Abstract:
"""

    return prompt

# Prepare Dataset for Training

In [ ]:
from datasets import Dataset

train["prompt"] = train["latex"].apply(build_prompt)

dataset = Dataset.from_pandas(
    train[["prompt", "abstract"]]
)

# Load the Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Set padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

# Tokenization 

In [ ]:
def tokenize(example):

    inputs = tokenizer(
        example["prompt"],
        max_length=1024,
        truncation=True
    )

    labels = tokenizer(
        example["abstract"],
        max_length=256,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset.set_format("torch")

# Training Setup

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    num_train_epochs=50,
    dataloader_pin_memory=False,
    save_strategy="no",
    report_to="none"
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# Train the Model

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

# Generate the Abstracts

In [ ]:
import torch
from tqdm import tqdm

def generate_abstract(latex):

    prompt = build_prompt(latex)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=180,
        num_beams=5,
        length_penalty=1.0,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Run on Test Data

In [ ]:
predictions = []

for latex in tqdm(test["latex"]):

    abstract = generate_abstract(latex)

    predictions.append(abstract)

# Create the Submission

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "abstract": predictions
})
submission.to_csv("working/submission.csv", index=False)